In [12]:
# retina face for face cropping
import os
import cv2
from tqdm import tqdm
from retinaface import RetinaFace

INPUT_ROOT = "dataset_split/unknown/clean_images"
OUTPUT_ROOT = "dataset_split/unknown/cropped_faces"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

total = 0

for identity in tqdm(os.listdir(INPUT_ROOT)):

    input_dir = os.path.join(INPUT_ROOT, identity)
    output_dir = os.path.join(OUTPUT_ROOT, identity)

    os.makedirs(output_dir, exist_ok=True)

    for image_file in os.listdir(input_dir):

        image_path = os.path.join(input_dir, image_file)

        img = cv2.imread(image_path)

        if img is None:
            continue

        try:

            faces = RetinaFace.detect_faces(image_path)

            if not isinstance(faces, dict):
                continue

            best_face = max(faces.values(), key=lambda x: x["score"])

            x1, y1, x2, y2 = best_face["facial_area"]

            face_crop = img[y1:y2, x1:x2]

            save_path = os.path.join(output_dir, image_file)

            cv2.imwrite(save_path, face_crop)

            total += 1

        except:
            continue

print("Total Cropped Faces:", total)

100%|██████████| 20/20 [00:49<00:00,  2.49s/it]

Total Cropped Faces: 26


In [13]:
# generating degraded images
import os
import cv2
from tqdm import tqdm

INPUT_ROOT = "dataset_split/unknown/cropped_faces"
OUTPUT_ROOT = "dataset_split/unknown/degraded_probes"

os.makedirs(OUTPUT_ROOT, exist_ok=True)


def blur_low(img):
    return cv2.GaussianBlur(img, (5, 5), 0)


def blur_high(img):
    return cv2.GaussianBlur(img, (11, 11), 0)


def jpeg_low(img):
    _, enc = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), 50])
    return cv2.imdecode(enc, 1)


def jpeg_high(img):
    _, enc = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), 15])
    return cv2.imdecode(enc, 1)


for identity in tqdm(os.listdir(INPUT_ROOT)):

    input_dir = os.path.join(INPUT_ROOT, identity)
    output_dir = os.path.join(OUTPUT_ROOT, identity)

    os.makedirs(output_dir, exist_ok=True)

    for image_file in os.listdir(input_dir):

        image_path = os.path.join(input_dir, image_file)

        img = cv2.imread(image_path)

        if img is None:
            continue

        base = os.path.splitext(image_file)[0]

        cv2.imwrite(os.path.join(output_dir, f"{base}_blur_low.jpg"), blur_low(img))

        cv2.imwrite(os.path.join(output_dir, f"{base}_blur_high.jpg"), blur_high(img))

        cv2.imwrite(os.path.join(output_dir, f"{base}_jpeg_low.jpg"), jpeg_low(img))

        cv2.imwrite(os.path.join(output_dir, f"{base}_jpeg_high.jpg"), jpeg_high(img))

print("Degradation Complete")

100%|██████████| 20/20 [00:00<00:00, 262.58it/s]

Degradation Complete


In [14]:
# generating arcface embeddings
import os
import cv2
import numpy as np

from tqdm import tqdm
from insightface.app import FaceAnalysis

INPUT_ROOT = "dataset_split/unknown/degraded_probes"
OUTPUT_ROOT = "embeddings/unknown/degraded_probe_embeddings"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Loading ArcFace...")

app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))

rec_model = app.models["recognition"]

print("ArcFace Ready")

count = 0

for identity in tqdm(os.listdir(INPUT_ROOT)):

    input_dir = os.path.join(INPUT_ROOT, identity)
    output_dir = os.path.join(OUTPUT_ROOT, identity)

    os.makedirs(output_dir, exist_ok=True)

    for image_file in os.listdir(input_dir):

        image_path = os.path.join(input_dir, image_file)

        img = cv2.imread(image_path)

        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (112, 112))

        embedding = rec_model.get_feat(img).flatten().astype(np.float32)

        save_path = os.path.join(output_dir, os.path.splitext(image_file)[0] + ".npy")

        np.save(save_path, embedding)

        count += 1

print("Total Embeddings:", count)

Loading ArcFace...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 

100%|██████████| 20/20 [00:10<00:00,  1.90it/s]

Total Embeddings: 104


In [ ]:
# creating unknown svm dataset in .csv format to feed to the model
import os
import joblib
import torch
import pyiqa
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

ROOT_IMAGES = "dataset_split"
ROOT_EMBEDDINGS = "embeddings"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

musiq_metric = pyiqa.create_metric("musiq", device=device)


def get_quality_score(image_path):

    score = musiq_metric(image_path)

    return float(score.cpu().item())


def load_gallery_db(gallery_root):

    gallery_db = {}

    for identity in os.listdir(gallery_root):

        gallery_db[identity] = []

        identity_dir = os.path.join(gallery_root, identity)

        for file in os.listdir(identity_dir):

            emb = np.load(os.path.join(identity_dir, file))

            gallery_db[identity].append(emb)

    return gallery_db


def search_gallery(probe_embedding, gallery_db):

    scores = {}

    for identity in gallery_db:

        sims = []

        for gallery_emb in gallery_db[identity]:

            sim = cosine_similarity(
                probe_embedding.reshape(1, -1), gallery_emb.reshape(1, -1)
            )[0][0]

            sims.append(sim)

        scores[identity] = max(sims)

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    predicted_identity = sorted_scores[0][0]

    best_similarity = sorted_scores[0][1]

    second_similarity = sorted_scores[1][1]

    margin = best_similarity - second_similarity

    return (predicted_identity, best_similarity, second_similarity, margin)


gallery_db = load_gallery_db("embeddings/test/gallery")

rows = []

probe_emb_root = "embeddings/unknown/degraded_probe_embeddings"

probe_img_root = "dataset_split/unknown/degraded_probes"

for identity in tqdm(os.listdir(probe_emb_root)):

    emb_dir = os.path.join(probe_emb_root, identity)

    img_dir = os.path.join(probe_img_root, identity)

    for emb_file in os.listdir(emb_dir):

        emb_path = os.path.join(emb_dir, emb_file)

        probe_embedding = np.load(emb_path)

        base_name = os.path.splitext(emb_file)[0]

        image_path = os.path.join(img_dir, base_name + ".jpg")

        quality_score = get_quality_score(image_path)

        predicted_identity, best_similarity, second_similarity, margin = search_gallery(
            probe_embedding, gallery_db
        )

        rows.append(
            {
                "unknown_identity": identity,
                "closest_match": predicted_identity,
                "quality_score": quality_score,
                "best_similarity": best_similarity,
                "second_similarity": second_similarity,
                "margin": margin,
                "label": 0,
            }
        )

unknown_df = pd.DataFrame(rows)

unknown_df.to_csv("unknown_svm_dataset.csv", index=False)

print(unknown_df.shape)

Loading pretrained model MUSIQ from /Users/admin/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth


100%|██████████| 20/20 [00:21<00:00,  1.08s/it]

(104, 7)


In [17]:
# run saved svm
import joblib
import pandas as pd

svm = joblib.load("svm_rejection_model.pkl")

scaler = joblib.load("svm_scaler.pkl")

df = pd.read_csv("unknown_svm_dataset.csv")

X = df[["quality_score", "best_similarity", "margin"]]

X_scaled = scaler.transform(X)

df["svm_decision"] = svm.predict(X_scaled)

df["confidence_score"] = svm.predict_proba(X_scaled)[:, 1]

df["final_decision"] = np.where(df["svm_decision"] == 1, df["closest_match"], "REJECT")

df.to_csv("unknown_results.csv", index=False)

total = len(df)

false_accepts = (df["svm_decision"] == 1).sum()

correct_rejects = (df["svm_decision"] == 0).sum()

TRR = correct_rejects / total
FAR = false_accepts / total

print("\nUNKNOWN TEST RESULTS")

print("Total:", total)
print("Correct Rejects:", correct_rejects)
print("False Accepts:", false_accepts)
print("TRR:", round(TRR, 4))
print("FAR:", round(FAR, 4))
from sklearn.metrics import confusion_matrix

y_true = np.zeros(len(df))
y_pred = df["svm_decision"]

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0,1]
)

print("\nConfusion Matrix")
print(cm)

TN = cm[0][0]
FP = cm[0][1]

print("\nTN:", TN)
print("FP:", FP)


UNKNOWN TEST RESULTS
Total: 104
Correct Rejects: 104
False Accepts: 0
TRR: 1.0
FAR: 0.0

Confusion Matrix
[[104   0]
 [  0   0]]

TN: 104
FP: 0


In [18]:
# verifying is resutls are legitimate
print("\n===== SIMILARITY STATS =====")
print(
    df["best_similarity"]
    .describe()
)

print("\n===== MARGIN STATS =====")
print(
    df["margin"]
    .describe()
)

print("\n===== CONFIDENCE STATS =====")
print(
    df["confidence_score"]
    .describe()
)


===== SIMILARITY STATS =====
count    104.000000
mean       0.212719
std        0.040814
min        0.131846
25%        0.184354
50%        0.210809
75%        0.231354
max        0.334046
Name: best_similarity, dtype: float64

===== MARGIN STATS =====
count    104.000000
mean       0.029992
std        0.026570
min        0.000070
25%        0.009240
50%        0.023416
75%        0.040592
max        0.123118
Name: margin, dtype: float64

===== CONFIDENCE STATS =====
count    104.000000
mean       0.008141
std        0.008931
min        0.002426
25%        0.003269
50%        0.005630
75%        0.009194
max        0.076045
Name: confidence_score, dtype: float64


In [22]:
#fixed threshold testing
# fixed threshold testing

THRESHOLD = 0.6

threshold_decision = (
    df["best_similarity"] >= THRESHOLD
).astype(int)

threshold_false_accepts = (
    threshold_decision == 1
).sum()

threshold_correct_rejects = (
    threshold_decision == 0
).sum()

threshold_TRR = (
    threshold_correct_rejects
    /
    len(df)
)

threshold_FAR = (
    threshold_false_accepts
    /
    len(df)
)

print("\n===== FIXED THRESHOLD RESULTS =====")

print("Threshold:", THRESHOLD)

print("Correct Rejects:", threshold_correct_rejects)

print("False Accepts:", threshold_false_accepts)

print("TRR:", round(threshold_TRR,4))

print("FAR:", round(threshold_FAR,4))


===== FIXED THRESHOLD RESULTS =====
Threshold: 0.6
Correct Rejects: 104
False Accepts: 0
TRR: 1.0
FAR: 0.0


In [23]:
#confusion matrix for fixed threshold
from sklearn.metrics import confusion_matrix

y_true = np.zeros(len(df))

cm_threshold = confusion_matrix(
    y_true,
    threshold_decision,
    labels=[0,1]
)

print("\nThreshold Confusion Matrix")

print(cm_threshold)


Threshold Confusion Matrix
[[104   0]
 [  0   0]]


In [24]:
# comapriosn table for both the methods
comparison = pd.DataFrame({

    "Method":[
        "Fixed Threshold",
        "SVM"
    ],

    "TRR":[
        threshold_TRR,
        TRR
    ],

    "FAR":[
        threshold_FAR,
        FAR
    ],

    "Correct Rejects":[
        threshold_correct_rejects,
        correct_rejects
    ],

    "False Accepts":[
        threshold_false_accepts,
        false_accepts
    ]

})

print(comparison)

            Method  TRR  FAR  Correct Rejects  False Accepts
0  Fixed Threshold  1.0  0.0              104              0
1              SVM  1.0  0.0              104              0
